In [ ]:
import pandas as pd
import numpy as np
from mp_api.client import MPRester

In [ ]:
nanozyme = pd.read_excel('./data/catalase.xlsx')

In [ ]:
nanozyme = nanozyme[['formula', 'space', 'pearson', 'volume', 'density']]
nanozyme['space'] = nanozyme['space'].str.extract(r'\((\d+)\)')[0].astype(float)

In [ ]:
def get_crystal_system(symbol: str) -> str:
    if symbol.startswith('c'):
        return 'cubic'
    elif symbol.startswith('hR'):
        return 'trigonal'
    elif symbol.startswith('hP'):
        return 'hexagonal'
    elif symbol.startswith(('tP', 'tI', 'tC')):
        return 'tetragonal'
    elif symbol.startswith('o'):
        return 'orthorhombic'
    elif symbol.startswith('m'):
        return 'monoclinic'
    elif symbol.startswith('a'):
        return 'triclinic'
    else:
        return None

nanozyme['pearson'] = nanozyme['pearson'].astype(str).apply(get_crystal_system).str.lower()

In [ ]:
nanozyme['volume'] = nanozyme['volume'].replace(" ", np.nan)
nanozyme['density'] = nanozyme['density'].replace(" ", np.nan)

nanozyme['volume'] = nanozyme['volume'].astype(float)
nanozyme['density'] = nanozyme['density'].astype(float)

In [ ]:
api_key = 'm62Lul7S3XVZCFIqkJA3PnOOwqWzpEWy'

docs = []
with MPRester(api_key) as mpr:
    for f in nanozyme.formula.tolist():
        res = mpr.summary.search(elements=[], formula=f, fields=['material_id', 'formula_pretty', 'theoretical',
                                                                 'symmetry', 'volume', 'density',
                                                                 'band_gap', 'energy_above_hull','efermi', 'formation_energy_per_atom'])
        docs.append(res)

In [ ]:
rows = []

for i in range(len(docs)):
    space = nanozyme.iloc[i]['space']
    pearson = nanozyme.iloc[i]['pearson']

    candidates = []
    row = nanozyme.iloc[i]
    
    candidates = [d for d in docs[i] if d.symmetry.number == space]
    if not candidates:
        candidates = [d for d in docs[i] if d.symmetry.crystal_system.value == pearson]
    if not candidates:
        candidates = docs[i]

    if candidates:
        non_theoretical = [d for d in candidates if d.theoretical == False]
        if non_theoretical:
            best = min(non_theoretical, key=lambda d: d.energy_above_hull)
        else:
            best = min(candidates, key=lambda d: d.energy_above_hull)

        rows.append({
            "formula": row['formula'],
            "pearson": row['pearson'] if pd.notna(row.get('pearson')) else best.symmetry.crystal_system.value,
            "space": row['space'] if pd.notna(row.get('space')) else best.symmetry.number,
            "volume": row['volume'] if pd.notna(row.get('volume')) else best.volume,
            "density": row['density'] if pd.notna(row.get('density')) else best.density,
            "band_gap": best.band_gap,
            "efermi": best.efermi,
            "formation_energy_per_atom": best.formation_energy_per_atom,
        })
    else:
        rows.append({
            "formula": row['formula'],
            "pearson": row['pearson'],
            "space": row['space'],
            "volume": row['volume'],
            "density": row['density'],
            "band_gap": None,
            "efermi": None,
            "formation_energy_per_atom": None,
        })

nanozyme_v2 = pd.DataFrame(rows)

In [ ]:
nanozyme_v2.to_csv('./data/catalase_features.csv', index=False)